In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, LeaveOneOut
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [10]:
df = pd.read_csv(r"D:\Foundathon3.0\Mess-Food-Analytics\Final_Data\final_training_data.csv")

df.head()

,Mess,Month,Meal,Waste_KG,Plates,Waste_Percent,Estimated_Dish
0,D-MESS,Apr-23,BF,1006.0,24455,4,Idli/Dosa/Poha/Bread/Pongal/Chappathi/Vada/Upm...
1,D-MESS,Apr-23,LUNCH,3161.0,31425,10,Rice/Dal/Sabzi/Curd/Palak Panneer/Chicken Curr...
2,D-MESS,Apr-23,SNACKS,0.0,31350,0,"Tea/Biscuits/Samosa/Pav Baji,Pani Poori/Bonda"
3,D-MESS,Apr-23,DINNER,3175.0,31360,10,Roti/Paneer/Chicken/Rice/Jeera Dal/Panjabi Par...
4,D-MESS,May-23,BF,1023.0,24240,4,Idli/Dosa/Poha/Bread/Pongal/Chappathi/Vada/Upm...


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 288 entries, 0 to 287
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Mess            288 non-null    object 
 1   Month           288 non-null    object 
 2   Meal            288 non-null    object 
 3   Waste_KG        288 non-null    float64
 4   Plates          288 non-null    int64  
 5   Waste_Percent   288 non-null    int64  
 6   Estimated_Dish  288 non-null    object 
dtypes: float64(1), int64(2), object(4)
memory usage: 15.9+ KB


In [12]:
df.describe()

,Waste_KG,Plates,Waste_Percent
count,288.000000,288.000000,288.000000
mean,1111.520833,34986.677083,3.552083
std,1724.169786,39191.348092,3.458675
min,0.000000,0.000000,0.000000
25%,0.000000,11430.250000,0.000000
50%,657.500000,22993.500000,3.000000
75%,1235.000000,42316.750000,5.000000
max,11785.000000,171598.000000,19.000000


In [13]:
df.columns

Index(['Mess', 'Month', 'Meal', 'Waste_KG', 'Plates', 'Waste_Percent',
       'Estimated_Dish'],
      dtype='object')

In [15]:
df = df[df['Plates'] > 0].reset_index(drop=True)

print(f"Loaded {len(df)} rows after removing closed-mess entries")

Loaded 286 rows after removing closed-mess entries


Feature Engineering

In [16]:
df['month_parsed'] = pd.to_datetime(df['Month'], format='%b-%y')
df['month_num'] = df['month_parsed'].dt.month       # 1–12, captures seasonality
df['year']      = df['month_parsed'].dt.year        # 2023 or 2024

In [17]:
# Sem 1: Jun–Nov | Sem 2: Dec–May
df['semester'] = df['month_num'].apply(lambda m: 1 if m >= 6 else 2)

In [18]:
df['is_holiday_month'] = df['month_num'].isin([5, 12]).astype(int)  # May, December

In [21]:
def clean_dishes(dish_str):
    """Split on / and clean each dish name."""
    if not isinstance(dish_str, str):
        return []
    return [d.strip().lower().replace(',', '') for d in dish_str.split('/') if d.strip()]

df['dish_list'] = df['Estimated_Dish'].apply(clean_dishes)

mlb = MultiLabelBinarizer()
dish_encoded = pd.DataFrame(
    mlb.fit_transform(df['dish_list']),
    columns=[f'dish_{d.replace(" ", "_")}' for d in mlb.classes_],
    index=df.index
)

print(f"Encoded {len(mlb.classes_)} unique dishes as features")

Encoded 31 unique dishes as features


In [22]:
df.head()

,Mess,Month,Meal,Waste_KG,Plates,Waste_Percent,Estimated_Dish,month_parsed,month_num,year,semester,is_holiday_month,dish_list
0,D-MESS,Apr-23,BF,1006.0,24455,4,Idli/Dosa/Poha/Bread/Pongal/Chappathi/Vada/Upm...,2023-04-01,4,2023,2,0,"[idli, dosa, poha, bread, pongal, chappathi, v..."
1,D-MESS,Apr-23,LUNCH,3161.0,31425,10,Rice/Dal/Sabzi/Curd/Palak Panneer/Chicken Curr...,2023-04-01,4,2023,2,0,"[rice, dal, sabzi, curd, palak panneer, chicke..."
2,D-MESS,Apr-23,SNACKS,0.0,31350,0,"Tea/Biscuits/Samosa/Pav Baji,Pani Poori/Bonda",2023-04-01,4,2023,2,0,"[tea, biscuits, samosa, pav bajipani poori, bo..."
3,D-MESS,Apr-23,DINNER,3175.0,31360,10,Roti/Paneer/Chicken/Rice/Jeera Dal/Panjabi Par...,2023-04-01,4,2023,2,0,"[roti, paneer, chicken, rice, jeera dal, panja..."
4,D-MESS,May-23,BF,1023.0,24240,4,Idli/Dosa/Poha/Bread/Pongal/Chappathi/Vada/Upm...,2023-05-01,5,2023,2,1,"[idli, dosa, poha, bread, pongal, chappathi, v..."


In [23]:
meal_dummies = pd.get_dummies(df['Meal'], prefix='meal')
mess_dummies = pd.get_dummies(df['Mess'], prefix='mess')

In [24]:
feature_df = pd.concat([
    df[['month_num', 'year', 'semester', 'is_holiday_month']],
    meal_dummies,
    mess_dummies,
    dish_encoded
], axis=1)

target = df['Plates']

In [25]:
print(f"Feature matrix shape: {feature_df.shape}")
print(f"   → Time features     : month_num, year, semester, is_holiday_month")
print(f"   → Meal dummies      : {list(meal_dummies.columns)}")
print(f"   → Mess dummies      : {list(mess_dummies.columns)}")
print(f"   → Dish features     : {len(dish_encoded.columns)} columns")

Feature matrix shape: (286, 45)
   → Time features     : month_num, year, semester, is_holiday_month
   → Meal dummies      : ['meal_BF', 'meal_DINNER', 'meal_LUNCH', 'meal_SNACKS']
   → Mess dummies      : ['mess_D-MESS', 'mess_J-MESS', 'mess_MEENAKSHI', 'mess_SANNASI', 'mess_SENBAGAM', 'mess_STAFF MESS']
   → Dish features     : 31 columns


Model Training

In [27]:
X = feature_df
y = target



models = {
    'Random Forest'       : RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=2),
    'Gradient Boosting'   : GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, random_state=42),
}

In [28]:
results = {}
for name, model in models.items():
    mae_scores  = -cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error')
    r2_scores   =  cross_val_score(model, X, y, cv=5, scoring='r2')

    results[name] = {
        'MAE_mean' : mae_scores.mean(),
        'MAE_std'  : mae_scores.std(),
        'R2_mean'  : r2_scores.mean(),
        'R2_std'   : r2_scores.std(),
    }

    print(f"\n{name}")
    print(f"  MAE  : {mae_scores.mean():,.0f} ± {mae_scores.std():,.0f} plates")
    print(f"  R²   : {r2_scores.mean():.3f} ± {r2_scores.std():.3f}")


Random Forest
  MAE  : 13,267 ± 5,052 plates
  R²   : 0.117 ± 0.970

Gradient Boosting
  MAE  : 12,640 ± 4,903 plates
  R²   : 0.230 ± 0.813


In [29]:
best_name = min(results, key=lambda k: results[k]['MAE_mean'])
print(f"\nBest model: {best_name}")

best_model = models[best_name]
best_model.fit(X, y)


Best model: Gradient Boosting


,loss,'squared_error'
,learning_rate,0.05
,n_estimators,200
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


Feature importance

In [30]:
importances = pd.Series(best_model.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(15)

print("\n" + "="*55)
print("TOP 15 MOST IMPORTANT FEATURES")
print("="*55)
for feat, imp in top_features.items():
    bar = '█' * int(imp * 200)
    print(f"  {feat:<35} {imp:.4f}  {bar}")


TOP 15 MOST IMPORTANT FEATURES
  mess_SANNASI                        0.6174  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  month_num                           0.2057  █████████████████████████████████████████
  mess_MEENAKSHI                      0.0249  ████
  mess_STAFF MESS                     0.0184  ███
  dish_pongal                         0.0167  ███
  dish_chappathi                      0.0121  ██
  dish_poori                          0.0119  ██
  meal_BF                             0.0116  ██
  semester                            0.0112  ██
  dish_dosa                           0.0105  ██
  dish_upma                           0.0104  ██
  dish_vada                           0.0095  █
  dish_bread                          0.0085  █
  mess_J-MESS                         0.0055  █
  dish_idli                           0.0053  █


In [31]:
def predict_plates(mess, meal, month_str, dishes):
    """
    Predict plate count for a given mess/meal/month/dishes combo.

    Parameters:
        mess      : str  e.g. 'D-MESS'
        meal      : str  e.g. 'LUNCH'
        month_str : str  e.g. 'Aug-24'
        dishes    : list e.g. ['rice', 'dal', 'chicken', 'curd']

    Returns:
        Predicted plate count (int)
    """
    row = pd.DataFrame(columns=X.columns, data=[[0]*len(X.columns)])

    # Time features
    dt = pd.to_datetime(month_str, format='%b-%y')
    row['month_num']        = dt.month
    row['year']             = dt.year
    row['semester']         = 1 if dt.month >= 6 else 2
    row['is_holiday_month'] = int(dt.month in [5, 12])

    # Meal & Mess
    meal_col = f'meal_{meal}'
    mess_col = f'mess_{mess}'
    if meal_col in row.columns: row[meal_col] = 1
    if mess_col in row.columns: row[mess_col] = 1

    # Dishes
    for d in dishes:
        col = f'dish_{d.strip().lower().replace(" ", "_")}'
        if col in row.columns:
            row[col] = 1

    pred = best_model.predict(row)[0]
    return max(0, int(round(pred)))

In [38]:

print("\nSAMPLE PREDICTIONS")


examples = [
    ('D-MESS',  'LUNCH',  'Mar-24', ['rice', 'dal', 'chicken', 'curd', 'sabzi']),
    ('J-MESS',  'BF',     'Oct-23', ['idli', 'dosa', 'poha', 'bread']),
    ('SANNASI', 'DINNER', 'Jan-24', ['roti', 'paneer', 'rice', 'jeera dal']),
    ('D-MESS',  'SNACKS', 'Dec-23', ['tea', 'biscuits', 'samosa']),
    ('STAFF MESS',  'LUNCH', 'Mar-24', ['roti', 'paneer', 'chicken']),
]

for mess, meal, month, dishes in examples:
    pred = predict_plates(mess, meal, month, dishes)
    print(f"  {mess:<12} | {meal:<7} | {month} | {', '.join(dishes[:3])}...")
    print(f"  → Predicted plates: {pred:,}\n")

print("Pipeline complete. Model ready for use.")


SAMPLE PREDICTIONS
  D-MESS       | LUNCH   | Mar-24 | rice, dal, chicken...
  → Predicted plates: 42,811

  J-MESS       | BF      | Oct-23 | idli, dosa, poha...
  → Predicted plates: 34,962

  SANNASI      | DINNER  | Jan-24 | roti, paneer, rice...
  → Predicted plates: 124,224

  D-MESS       | SNACKS  | Dec-23 | tea, biscuits, samosa...
  → Predicted plates: 12,570

  STAFF MESS   | LUNCH   | Mar-24 | roti, paneer, chicken...
  → Predicted plates: 17,892

Pipeline complete. Model ready for use.
